In [3]:
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

In [4]:
# 데이터 불러오기
data = pd.read_csv("mushroom.csv")

# 데이터 확인
data.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [5]:
# 데이터 크기 확인
data.shape

(8124, 23)

In [6]:
# 결측값 확인
data.isnull().sum()

class                       0
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises                     0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
dtype: int64

In [7]:
# 독립변수 / 종속변수 분리
X = data.drop("class", axis=1)
y = data["class"]

print(X.shape)
print(y.shape)

(8124, 22)
(8124,)


In [8]:
# 범주형 데이터를 숫자형으로 변환
X = pd.get_dummies(X, dtype=int)

# 변환 결과 확인
X.head()

,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_f,cap-surface_g,cap-surface_s,cap-surface_y,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,0,0,0,0,0,1,0,0,1,0,...,1,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
2,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,1,0,0,0,1,...,1,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0


In [9]:
X

,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_f,cap-surface_g,cap-surface_s,cap-surface_y,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,0,0,0,0,0,1,0,0,1,0,...,1,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
2,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,1,0,0,0,1,...,1,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
8120,0,0,0,0,0,1,0,0,1,0,...,0,1,0,0,0,1,0,0,0,0
8121,0,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
8122,0,0,0,1,0,0,0,0,0,1,...,0,1,0,0,0,1,0,0,0,0


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (6499, 117)
X_test: (1625, 117)
y_train: (6499,)
y_test: (1625,)


In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# 의사결정트리 모델 생성
dt = DecisionTreeClassifier(random_state=42)

# 탐색할 파라미터 설정
dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "splitter": ["best", "random"],
    "max_depth": [None, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_split": [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_leaf": [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "max_leaf_nodes": [None, 5, 10, 15, 20, 30]
}

# GridSearchCV 설정
dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# 학습
dt_grid.fit(X_train, y_train)

# 최적 파라미터 확인
print("Decision Tree 최적 파라미터")
print(dt_grid.best_params_)


# 최적 모델 저장
best_dt = dt_grid.best_estimator_

# 테스트 데이터 예측
dt_pred = best_dt.predict(X_test)

# 테스트 정확도
dt_accuracy = accuracy_score(y_test, dt_pred)

print("Decision Tree 테스트 정확도:", dt_accuracy)

Decision Tree 최적 파라미터
{'criterion': 'gini', 'max_depth': None, 'max_leaf_nodes': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'splitter': 'best'}
Decision Tree 테스트 정확도: 1.0


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# 랜덤포레스트 모델 생성
rf = RandomForestClassifier(random_state=42)

# 탐색할 파라미터 설정
rf_param_grid = {
    "n_estimators": [50, 100],
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_split": [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_leaf": [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "max_leaf_nodes": [None, 5, 10, 15, 20, 30]
}

# GridSearchCV 설정
rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# 학습
rf_grid.fit(X_train, y_train)

# 최적 파라미터 확인
print("Random Forest 최적 파라미터")
print(rf_grid.best_params_)

# 최적 모델 저장
best_rf = rf_grid.best_estimator_

# 테스트 데이터 예측
rf_pred = best_rf.predict(X_test)

# 테스트 정확도
rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest 테스트 정확도:", rf_accuracy)

Random Forest 최적 파라미터
{'criterion': 'gini', 'max_depth': None, 'max_leaf_nodes': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Random Forest 테스트 정확도: 1.0
